In [1]:
!pip install -q -U "transformers[serving]" accelerate bitsandbytes

In [2]:
import os
from kaggle_secrets import UserSecretsClient
MODEL = "xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic"   # EDIT: exact HF repo id, org/name, safetensors, no colon
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")   # only for gated models
# assert "/" in MODEL and MODEL != "xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic", "set MODEL first"

from huggingface_hub import snapshot_download
path = snapshot_download(MODEL, ignore_patterns=["*.gguf", "*.pth", "original/*"])
print("downloaded:", path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

downloaded: /root/.cache/huggingface/hub/models--xtools-at--DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic/snapshots/fbccdc6adf70a1981e8f40e91a8370e428e4f998


In [3]:
import os, subprocess, time, requests
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
PROMPT_WORDS = 2500   # stress test size. Set near your real prompt (chars / 4 = tokens)

serve = subprocess.Popen(
    ["transformers", "serve", "--port", "8000",
     "--quantization", "bnb-4bit",
     "--model-timeout", "43200"],   # 12h idle before unload, so no reload mid-session
    stdout=open("serve.log", "w"), stderr=subprocess.STDOUT)

for _ in range(120):
    try:
        requests.get("http://localhost:8000/v1/models", timeout=2)
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("server did not start:\n" + open("serve.log").read()[-3000:])

def chat(text, max_tokens=8):
    t = time.time()
    r = requests.post("http://localhost:8000/v1/chat/completions",
        json={"model": MODEL, "messages": [{"role": "user", "content": text}], "max_tokens": max_tokens},
        timeout=1800)
    print(f"  {r.status_code} in {time.time() - t:.0f}s")
    if r.status_code != 200:
        raise RuntimeError(r.text[:500] + "\n--- serve.log ---\n" + open("serve.log").read()[-3000:])

print("loading model into GPU...")
chat("hi")
print(f"stress test, ~{PROMPT_WORDS} word prompt...")
chat("word " * PROMPT_WORDS)
print("READY. Model loaded and survived long prompt.")
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

loading model into GPU...
  200 in 60s
stress test, ~2500 word prompt...
  200 in 11s
READY. Model loaded and survived long prompt.
memory.used [MiB], memory.total [MiB]
5001 MiB, 15360 MiB
5415 MiB, 15360 MiB


In [4]:
import re, time, subprocess
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stderr=open("cf.log", "w"),
)
for _ in range(30):
    time.sleep(1)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("cf.log").read())
    if m:
        URL = m.group(0)
        print("BASE URL:", URL + "/v1")
        break

BASE URL: https://kiss-portsmouth-unable-transport.trycloudflare.com/v1


In [5]:
!pip install -q fastapi uvicorn sentence-transformers

In [6]:
%%writefile embed_server.py
import base64, io, os, tempfile, threading
import torch
from fastapi import FastAPI
from pydantic import BaseModel
from PIL import Image
from sentence_transformers import SentenceTransformer
import uuid, time

app = FastAPI()
m = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cpu")
m.max_seq_length = 256

OCR_ID = "zai-org/GLM-OCR"
_ocr = {}
_lock = threading.Lock()

def ocr_model():
    if not _ocr:
        from transformers import AutoProcessor, AutoModelForImageTextToText
        _ocr["p"] = AutoProcessor.from_pretrained(OCR_ID)
        _ocr["m"] = AutoModelForImageTextToText.from_pretrained(
            OCR_ID, torch_dtype="auto", device_map="auto")
    return _ocr["p"], _ocr["m"]

class Req(BaseModel):
    model: str | None = None
    input: str | list[str]

class OcrReq(BaseModel):
    image: str
    prompt: str = "Text Recognition:"
    max_new_tokens: int = 2048

@app.post("/v1/embeddings")
def emb(r: Req):
    xs = [r.input] if isinstance(r.input, str) else r.input
    v = m.encode(xs, normalize_embeddings=True, batch_size=16)
    return {"object": "list", "model": "bge-small-en-v1.5",
            "data": [{"object": "embedding", "index": i, "embedding": e.tolist()} for i, e in enumerate(v)],
            "usage": {"prompt_tokens": 0, "total_tokens": 0}}

_jobs = {}

def _run(jid, r):
    try:
        img = Image.open(io.BytesIO(base64.b64decode(r.image))).convert("RGB")
        with _lock:
            p, mod = ocr_model()
            with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as f:
                img.save(f, "PNG")
                path = f.name
            try:
                msgs = [{"role": "user", "content": [
                    {"type": "image", "url": path},
                    {"type": "text", "text": r.prompt}]}]
                inputs = p.apply_chat_template(
                    msgs, add_generation_prompt=True, tokenize=True,
                    return_dict=True, return_tensors="pt").to(mod.device)
                inputs.pop("token_type_ids", None)
                with torch.inference_mode():
                    out = mod.generate(**inputs, max_new_tokens=r.max_new_tokens, do_sample=False)
                text = p.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            finally:
                os.unlink(path)
        _jobs[jid] = {"status": "done", "text": text.strip()}
    except Exception as e:
        _jobs[jid] = {"status": "error", "error": f"{type(e).__name__}: {e}"}

@app.post("/v1/ocr/start")
def ocr_start(r: OcrReq):
    jid = uuid.uuid4().hex
    _jobs[jid] = {"status": "running"}
    threading.Thread(target=_run, args=(jid, r), daemon=True).start()
    return {"id": jid}

@app.get("/v1/ocr/{jid}")
def ocr_poll(jid: str):
    return _jobs.get(jid, {"status": "missing"})

def _preload():
    with _lock:
        ocr_model()

threading.Thread(target=_preload, daemon=True).start()

Writing embed_server.py


In [7]:
subprocess.Popen(["uvicorn", "embed_server:app", "--port", "8001"],
                 stdout=open("embed.log", "w"), stderr=subprocess.STDOUT)
subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8001"],
                 stderr=open("cf2.log", "w"))
for _ in range(30):
    time.sleep(1)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("cf2.log").read())
    if m:
        print("EMBED URL:", m.group(0) + "/v1")
        break

EMBED URL: https://grad-mentor-our-decrease.trycloudflare.com/v1


In [8]:
import base64, io, time, requests
from PIL import Image, ImageDraw, ImageFont

for _ in range(60):
    try:
        requests.get("http://localhost:8001/docs", timeout=2)
        break
    except Exception:
        time.sleep(2)

img = Image.new("RGB", (700, 140), "white")
ImageDraw.Draw(img).text((20, 40), "OCR TEST 12345", fill="black", font=ImageFont.load_default(size=48))
buf = io.BytesIO()
img.save(buf, "PNG")
r = requests.post("http://localhost:8001/v1/ocr/start",
                  json={"image": base64.b64encode(buf.getvalue()).decode()}, timeout=30)
jid = r.json()["id"]
for _ in range(200):
    time.sleep(3)
    d = requests.get(f"http://localhost:8001/v1/ocr/{jid}", timeout=30).json()
    if d["status"] != "running":
        break
print(d)
print(r.status_code, r.text[:300])

{'status': 'done', 'text': 'OCR TEST 12345'}
200 {"id":"867a0f2869864b7a8827b2fb8ca55183"}
